In [1]:
# 1. Импорт библиотек
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv

print("✅ Библиотеки импортированы (актуальные версии 2025)")
print(f"Рабочая директория: {os.getcwd()}")

✅ Библиотеки импортированы (актуальные версии 2025)
Рабочая директория: d:\Projects\rag-assistant\notebooks


In [1]:
# 2. Альтернатива: используем локальные эмбеддинги (бесплатно)
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ Используем бесплатные эмбеддинги HuggingFace")

C:\Users\Китя\AppData\Local\Temp\ipykernel_8992\3792169565.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Китя\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Китя\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Используем бесплатные эмбеддинги HuggingFace


In [4]:
import os
print("Текущая папка:", os.getcwd())

# Поднимаемся в корень проекта
os.chdir("..")
print("Теперь мы в:", os.getcwd())
print("Содержит data?", "data" in os.listdir())

Текущая папка: d:\Projects\rag-assistant\notebooks
Теперь мы в: d:\Projects\rag-assistant
Содержит data? True


In [5]:
# 3. Загружаем PDF из папки data
import os
from langchain_community.document_loaders import PyPDFLoader

# Ищем PDF файлы в папке data
pdf_files = [f for f in os.listdir("data") if f.endswith(".pdf")]

if pdf_files:
    file_path = os.path.join("data", pdf_files[0])
    print(f"✅ Найден файл: {pdf_files[0]}")
    
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    print(f"✅ Загружено {len(documents)} страниц")
    print(f"📄 Пример текста: {documents[0].page_content[:200]}...")
else:
    print("❌ В папке data нет PDF файлов")
    print("Положи любой PDF в папку data/ и перезапусти ячейку")

✅ Найден файл: 2301.00234v6.pdf
✅ Загружено 22 страниц
📄 Пример текста: A Survey on In-context Learning
Qingxiu Dong1, Lei Li1, Damai Dai1, Ce Zheng1, Jingyuan Ma1, Rui Li1, Heming Xia2,
Jingjing Xu3, Zhiyong Wu4, Tianyu Liu5, Baobao Chang1, Xu Sun1, Lei Li6 and Zhifang S...


In [7]:
# 4. Разбиваем текст на чанки
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"✅ Создано {len(chunks)} чанков")
print(f"📊 Пример чанка: {chunks[0].page_content[:100]}...")

✅ Создано 243 чанков
📊 Пример чанка: A Survey on In-context Learning
Qingxiu Dong1, Lei Li1, Damai Dai1, Ce Zheng1, Jingyuan Ma1, Rui Li1...


In [8]:
# 5. Создаём векторную базу данных
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Используем бесплатные эмбеддинги
embedding = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Создаём базу
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="./chroma_db"
)

print(f"✅ Векторная база создана")
print(f"  - Всего векторов: {vectordb._collection.count()}")
print(f"  - Папка: ./chroma_db")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Векторная база создана
  - Всего векторов: 243
  - Папка: ./chroma_db


In [9]:
# 6. Тестируем семантический поиск
question = "Что такое in-context learning?"  # вопрос по содержанию PDF

# Поиск по смыслу
results = vectordb.similarity_search_with_score(question, k=3)

print(f"🔍 Вопрос: {question}\n")
print("Найденные фрагменты (с оценкой релевантности):")

for i, (doc, score) in enumerate(results):
    print(f"\n--- Результат {i+1} (релевантность: {score:.4f}) ---")
    print(doc.page_content[:300] + "...")
    
print(f"\n📊 Всего найдено релевантных чанков: {len(results)}")

🔍 Вопрос: Что такое in-context learning?

Найденные фрагменты (с оценкой релевантности):

--- Результат 1 (релевантность: 0.7591) ---
Danny Hernandez, Scott Johnston, Andy Jones, Jack-
son Kernion, Liane Lovitt, Kamal Ndousse, Dario
Amodei, Tom Brown, Jack Clark, Jared Kaplan, Sam
McCandlish, and Chris Olah. 2022. In-context learn-
ing and induction heads. CoRR, abs/2209.11895.
OpenAI. 2023. GPT-4 technical report. CoRR,
abs/2303....

--- Результат 2 (релевантность: 0.7904) ---
ested in this area and shed light on future research.
2 Definition and Formulation
Following Brown et al. (2020), we here provide a
formal definition of in-context learning:
In-context learning is a paradigm that
allows language models to learn tasks
given only a few examples in the form of
demonstr...

--- Результат 3 (релевантность: 0.8393) ---
Da Huang, Denny Zhou, and Tengyu Ma. 2023b.
Larger language models do in-context learning dif-
ferently. CoRR, abs/2303.03846.
Noam Wies, Yoav Levine, and Amnon Shashua

In [11]:
# 7. Сохраняем информацию о базе для следующего этапа
import os

# Просто запоминаем путь к папке с базой
db_path = "./chroma_db"
print(f"✅ Векторная база сохранена в папке: {db_path}")
print(f"   Путь абсолютный: {os.path.abspath(db_path)}")
print("\n🎉 Этап 1 полностью завершён!")
print("➡️ Для загрузки базы в следующем ноутбуке используй:")
print('vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding)')

✅ Векторная база сохранена в папке: ./chroma_db
   Путь абсолютный: d:\Projects\rag-assistant\chroma_db

🎉 Этап 1 полностью завершён!
➡️ Для загрузки базы в следующем ноутбуке используй:
vectordb = Chroma(persist_directory="./chroma_db", embedding_function=embedding)
